# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [ ]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [ ]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [ ]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [ ]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

In [ ]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [ ]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [ ]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Nombre del algoritmo

<div style="border-left:5px solid #4C72B0;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

*[Explica en lenguaje llano cómo funciona el algoritmo y, después, de forma precisa: pseudocódigo o pasos, cómo se aplica al problema del viajante.]*

- **Tipo:** exacto / heurístico / aproximado / metaheurística.
- **Complejidad en tiempo:** ...  **en memoria:** ...
- **Propiedades:** ¿siempre encuentra el óptimo? ¿es determinista? ¿tiene garantía de calidad? ¿necesita distancias simétricas (grafo no dirigido)?
- **¿Hasta cuántas ciudades es razonable usarlo?**

</div>

**💻 CÓDIGO**

*[Sustituye `mi_algoritmo` por tu algoritmo, con un nombre único (por ejemplo, `fuerza_bruta`). Debe cumplir el contrato: recibe el grafo de ciudades `K` y devuelve `(ruta, coste)`. Borra este texto entre corchetes.]*

In [ ]:
def mi_algoritmo(K, inicio=None):
    # Contrato: recibe el grafo de ciudades K (K[a][b]["peso"] = distancia de a a b) y devuelve (ruta, coste):
    # `ruta` = lista con todas las ciudades, cada una una vez, en el orden en que se visitan (el viajante
    # vuelve después a la primera); `coste` = coste total del ciclo. Debe funcionar con K dirigido (nx.DiGraph)
    # y no dirigido (nx.Graph).
    ciudades = list(K.nodes)
    if inicio is not None:                       # empezar por la ciudad pedida
        ciudades.remove(inicio)
        ciudades.insert(0, inicio)
    ruta = ciudades                              # EJEMPLO MÍNIMO (sin pensar): visita en el orden de la lista
    return ruta, coste_ruta(K, ruta)

**Prueba rápida** con una versión pequeña (las 6 ciudades de Galicia).

*[Cambia los parámetros de `cargar_version` para probar otras versiones y borra este texto entre corchetes.]*

In [ ]:
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = mi_algoritmo(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(mi_algoritmo, K, memoria=True, inicio="Vigo"))

<div style="border-left:5px solid #55A868;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

*[Tabla con lo medido: versión, nº de ciudades, coste, tiempo y memoria. Hasta cuántas ciudades ha funcionado. Observaciones.]*

</div>

---